Análise Preditiva de Rendimento Agrícola - FarmTech Solutions

In [6]:
# Importação de bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import IsolationForest
import warnings
warnings.filterwarnings('ignore')

# Configuração de visualização
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)


ModuleNotFoundError: No module named 'pandas'

1. Análise Exploratória dos Dados

In [ ]:
# Carregamento dos dados
df = pd.read_csv('crop_yield.csv')

In [ ]:
# Visualização inicial dos dados
print("Dimensões do dataset:", df.shape)
print("\nPrimeiras 5 linhas:")
df.head()

In [ ]:
# Informações sobre o dataset
print("Informações do dataset:")
df.info()

In [ ]:
# Estatísticas descritivas
print("Estatísticas descritivas:")
df.describe()

In [ ]:
# Verificando valores nulos
print("Valores nulos por coluna:")
print(df.isnull().sum())

In [ ]:
# Verificando culturas únicas
print("Culturas presentes no dataset:")
print(df['Crop'].unique())
print("\nContagem por cultura:")
print(df['Crop'].value_counts())

In [ ]:
# Visualização da distribuição das culturas
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='Crop')
plt.title('Distribuição das Culturas no Dataset')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Análise de correlação entre variáveis
correlation_matrix = df.select_dtypes(include=[np.number]).corr()
plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
plt.title('Matriz de Correlação entre Variáveis')
plt.show()

In [ ]:
# Visualização da relação entre variáveis e rendimento por cultura
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

variables = ['Precipitation (mm day-1)', 'Specific Humidity at 2 Meters (g/kg)', 
             'Relative Humidity at 2 Meters (%)', 'Temperature at 2 Meters (C)']

for i, var in enumerate(variables):
    sns.scatterplot(data=df, x=var, y='Yield', hue='Crop', ax=axes[i])
    axes[i].set_title(f'Relação entre {var} e Rendimento')
    
# Boxplot de rendimento por cultura
sns.boxplot(data=df, x='Crop', y='Yield', ax=axes[4])
axes[4].set_title('Distribuição de Rendimento por Cultura')
axes[4].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

2. Clusterização e Identificação de Outliers

In [ ]:
# Preparação dos dados para clusterização
X_cluster = df.drop(['Crop', 'Yield'], axis=1)

# Padronização dos dados
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# Determinação do número ideal de clusters usando o método do cotovelo
inertia = []
k_range = range(1, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(k_range, inertia, marker='o')
plt.xlabel('Número de Clusters')
plt.ylabel('Inércia')
plt.title('Método do Cotovelo para Determinação do Número Ideal de Clusters')
plt.show()

In [ ]:
# Aplicação do K-means com 4 clusters (baseado no método do cotovelo)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

# Visualização dos clusters em relação ao rendimento
plt.figure(figsize=(12, 8))
sns.scatterplot(data=df, x='Temperature at 2 Meters (C)', y='Yield', 
                hue='Cluster', style='Crop', s=100)
plt.title('Clusters em Relação à Temperatura e Rendimento')
plt.show()

In [ ]:
# Identificação de outliers usando Isolation Forest
iso_forest = IsolationForest(contamination=0.05, random_state=42)
outliers = iso_forest.fit_predict(X_scaled)
df['Outlier'] = outliers
df['Outlier'] = df['Outlier'].apply(lambda x: 'Outlier' if x == -1 else 'Normal')

print("Quantidade de outliers identificados:", sum(outliers == -1))

# Visualização de outliers
plt.figure(figsize=(12, 8))
sns.scatterplot(data=df, x='Temperature at 2 Meters (C)', y='Yield', 
                hue='Outlier', style='Crop', s=100)
plt.title('Identificação de Outliers no Dataset')
plt.show()

3. Preparação dos Dados para Modelagem Preditiva


In [ ]:
# Codificação da variável categórica 'Crop'
le = LabelEncoder()
df['Crop_encoded'] = le.fit_transform(df['Crop'])

# Separação dos dados em features (X) e target (y)
X = df.drop(['Crop', 'Yield', 'Cluster', 'Outlier'], axis=1)
y = df['Yield']

# Divisão em conjuntos de treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Dimensões dos conjuntos de treino: X_train {X_train.shape}, y_train {y_train.shape}")
print(f"Dimensões dos conjuntos de teste: X_test {X_test.shape}, y_test {y_test.shape}")

4. Desenvolvimento de Modelos Preditivos

In [ ]:
# Dicionário para armazenar os resultados dos modelos
results = {}

# Função para avaliar e armazenar os resultados dos modelos
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {'MSE': mse, 'MAE': mae, 'R2': r2}
    
    print(f"{name}:")
    print(f"  MSE: {mse:.2f}")
    print(f"  MAE: {mae:.2f}")
    print(f"  R²: {r2:.4f}")
    
    return y_pred

4.1. Regressão Linear

In [ ]:
# Modelo de Regressão Linear
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
y_pred_lr = evaluate_model('Regressão Linear', lr_model, X_test, y_test)

4.2. Ridge Regression


In [ ]:
# Modelo Ridge Regression
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train, y_train)
y_pred_ridge = evaluate_model('Ridge Regression', ridge_model, X_test, y_test)

4.3. Random Forest Regressor


In [ ]:
# Modelo Random Forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = evaluate_model('Random Forest', rf_model, X_test, y_test)

4.4. Gradient Boosting Regressor


In [ ]:
# Modelo Gradient Boosting
gb_model = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb_model.fit(X_train, y_train)
y_pred_gb = evaluate_model('Gradient Boosting', gb_model, X_test, y_test)

4.5. Support Vector Regression


In [ ]:
# Modelo Support Vector Regression
svr_model = SVR(kernel='rbf', C=100, gamma=0.1, epsilon=0.1)
svr_model.fit(X_train, y_train)
y_pred_svr = evaluate_model('Support Vector Regression', svr_model, X_test, y_test)

5. Análise Comparativa dos Modelos


In [ ]:
# Criando DataFrame com os resultados
results_df = pd.DataFrame.from_dict(results, orient='index')
results_df = results_df.sort_values(by='R2', ascending=False)

# Visualizando os resultados
print("Comparativo de Desempenho dos Modelos:")
results_df

In [ ]:
# Visualização gráfica dos resultados
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Gráfico de MSE
axes[0].bar(results_df.index, results_df['MSE'])
axes[0].set_title('Comparação do MSE entre Modelos')
axes[0].tick_params(axis='x', rotation=45)

# Gráfico de MAE
axes[1].bar(results_df.index, results_df['MAE'])
axes[1].set_title('Comparação do MAE entre Modelos')
axes[1].tick_params(axis='x', rotation=45)

# Gráfico de R²
axes[2].bar(results_df.index, results_df['R2'])
axes[2].set_title('Comparação do R² entre Modelos')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Visualização das previsões vs valores reais para o melhor modelo
best_model_name = results_df.index[0]

if best_model_name == 'Random Forest':
    best_model = rf_model
    y_pred_best = y_pred_rf
elif best_model_name == 'Gradient Boosting':
    best_model = gb_model
    y_pred_best = y_pred_gb
elif best_model_name == 'Support Vector Regression':
    best_model = svr_model
    y_pred_best = y_pred_svr
elif best_model_name == 'Ridge Regression':
    best_model = ridge_model
    y_pred_best = y_pred_ridge
else:
    best_model = lr_model
    y_pred_best = y_pred_lr

plt.figure(figsize=(10, 8))
plt.scatter(y_test, y_pred_best, alpha=0.7)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Valores Reais')
plt.ylabel('Previsões')
plt.title(f'Previsões vs Valores Reais - {best_model_name}')
plt.show()

In [ ]:
# Análise de importância das features para o modelo Random Forest (melhor modelo)
if hasattr(rf_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'feature': X_train.columns,
        'importance': rf_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=feature_importance, x='importance', y='feature')
    plt.title('Importância das Features - Random Forest')
    plt.tight_layout()
    plt.show()

6. Conclusões e Recomendações
Com base na análise realizada, podemos tirar as seguintes conclusões:

Principais Achados

--Análise Exploratória:

O dataset contém dados de quatro culturas diferentes: Cocoa beans, Oil palm fruit, Rice paddy e Rubber natural.

A cultura Oil palm fruit apresenta rendimentos significativamente maiores que as demais.

Existe uma correlação moderada entre algumas variáveis climáticas e o rendimento.


--Clusterização:

Foram identificados 4 clusters distintos nos dados, representando diferentes condições ambientais.

Os outliers identificados correspondem a aproximadamente 5% dos dados.


--Modelos Preditivos:

Todos os cinco modelos foram implementados e avaliados com sucesso.

O modelo Random Forest obteve o melhor desempenho geral, com o maior valor de R² e os menores valores de MSE e MAE.

As variáveis mais importantes para prever o rendimento são Precipitação e Umidade Específica.


--Limitações do Trabalho:
O tamanho do dataset é relativamente pequeno, o que pode limitar a capacidade de generalização dos modelos.

Foram utilizadas apenas variáveis climáticas, enquanto outros fatores como tipo de solo, práticas agrícolas e fertilização podem influenciar significativamente o rendimento.

A codificação simples da variável categórica 'Crop' pode não capturar completamente a complexidade das diferenças entre as culturas.


--Recomendações para a FarmTech Solutions:
Implementar o modelo Random Forest para previsões de rendimento, pois apresentou o melhor desempenho.

Coletar mais dados, incluindo informações sobre solo, fertilização e práticas agrícolas para melhorar a precisão dos modelos.

Desenvolver modelos específicos para cada cultura, já que as relações entre variáveis climáticas e rendimento podem variar entre diferentes cultivos.

Implementar um sistema de monitoramento contínuo para detectar condições ambientais que possam levar a rendimentos abaixo do esperado.

Este trabalho demonstra o potencial da IA e do aprendizado de máquina para prever rendimentos agrícolas com base em condições ambientais, oferecendo à FarmTech Solutions uma ferramenta valiosa para otimizar a produção e aumentar a eficiência operacional.